# 🥇 XAUUSD ML Bot — Notebook 1
## Data Exploration + Feature Engineering + First ML Model

---

### 📌 What You Will Learn in This Notebook
1. Connect to MetaTrader 5 and pull real XAUUSD data
2. Explore and understand your data (like an ML engineer)
3. Engineer features from your trading knowledge
4. Train your FIRST ML model on real gold data
5. Evaluate how good it is

### 🧠 The ML Engineer Mindset
```
Raw Data → Clean Data → Features → Model → Evaluate → Improve
```
This is the loop you will run forever as an ML engineer.

---

## 🔧 CELL 1 — Install Everything You Need
Run this ONCE when you first set up the project.

In [ ]:
# Run this cell first — installs all required libraries
# In terminal: pip install these one by one if cell fails

import subprocess
packages = [
    'MetaTrader5',
    'pandas',
    'numpy',
    'matplotlib',
    'scikit-learn',
    'xgboost',
    'ta',           # technical analysis library
    'joblib',
    'seaborn'
]

for pkg in packages:
    subprocess.run(['pip', 'install', pkg, '-q'])
    print(f'✅ {pkg} installed')

print('\n🚀 All packages ready!')

## 📦 CELL 2 — Import All Libraries
Think of imports as loading your tools before starting work.

In [ ]:
# ============================================================
# IMPORTS — Load all your tools
# ============================================================

# Data handling
import pandas as pd                    # DataFrames — like Excel in Python
import numpy as np                     # Math operations on arrays

# Visualization
import matplotlib.pyplot as plt        # Plotting charts
import matplotlib.dates as mdates
import seaborn as sns                  # Beautiful statistical charts

# Technical Analysis
import ta                              # All indicators: RSI, ATR, MACD etc.

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.tree import DecisionTreeClassifier, export_text

# XGBoost — best model for financial data
from xgboost import XGBClassifier

# Save models
import joblib
import warnings
import os

warnings.filterwarnings('ignore')

# Chart styling
plt.style.use('dark_background')
sns.set_palette('husl')

print('✅ All libraries loaded successfully!')
print('You are ready to build your XAUUSD ML Bot 🚀')

---
## 📡 STEP 1 — GET YOUR XAUUSD DATA

We have **two options**:
- **Option A** — Connect to MetaTrader 5 directly (if MT5 is installed)
- **Option B** — Load from a CSV file (exported from MT5)

Run whichever applies to you.

In [ ]:
# ============================================================
# OPTION A — Connect to MetaTrader 5
# ============================================================
# Only run this if MT5 is installed on your computer
# Skip to OPTION B if you want to use a CSV file

USE_MT5 = False  # ← Change to True if MT5 is installed

if USE_MT5:
    import MetaTrader5 as mt5

    # Connect to MT5
    if not mt5.initialize():
        print('❌ MT5 initialization failed')
        print('Make sure MetaTrader5 is open and logged in')
    else:
        print('✅ Connected to MetaTrader5!')

        # Pull XAUUSD H1 data — last 5000 candles (~7 months)
        rates = mt5.copy_rates_from_pos('XAUUSD', mt5.TIMEFRAME_H1, 0, 5000)

        # Convert to DataFrame
        df_raw = pd.DataFrame(rates)
        df_raw['time'] = pd.to_datetime(df_raw['time'], unit='s')
        df_raw.set_index('time', inplace=True)

        # Save to CSV so you can use Option B next time
        df_raw.to_csv('data/XAUUSD_H1_raw.csv')
        print(f'✅ Pulled {len(df_raw)} candles from MT5')
        print(f'📅 Date range: {df_raw.index[0]} → {df_raw.index[-1]}')

        mt5.shutdown()

In [ ]:
# ============================================================
# OPTION B — Load from CSV file
# ============================================================
# HOW TO EXPORT FROM MT5:
# 1. Open MT5 → History Center
# 2. Select XAUUSD → H1
# 3. Export → Save as CSV
# 4. Put the file in your 'data' folder
#
# OR: If you already ran Option A above,
#     it saved a CSV — just load that.

# --------------------------------------------------------
# If you have your own CSV, change the path below
# --------------------------------------------------------
CSV_PATH = 'data/XAUUSD_H1_raw.csv'  # ← your file path here

# ----- DEMO MODE -----
# If you don't have a file yet, we create demo data
# so you can still run the full notebook and learn!

if not os.path.exists(CSV_PATH):
    print('⚠️  No data file found — generating realistic DEMO data')
    print('    This lets you run the full notebook right now!')
    print('    Replace with your real MT5 data when ready.\n')

    # Generate realistic XAUUSD-like price data
    np.random.seed(42)
    n = 8000  # 8000 hourly candles (~11 months)

    dates = pd.date_range(start='2024-01-01', periods=n, freq='h')

    # Simulate realistic gold price movement
    returns = np.random.normal(0.0001, 0.003, n)  # small mean drift, realistic vol
    trend = np.sin(np.linspace(0, 4*np.pi, n)) * 0.001  # add macro trend
    close = 2000 * np.exp(np.cumsum(returns + trend))

    # Build OHLCV
    high = close * (1 + np.abs(np.random.normal(0, 0.002, n)))
    low  = close * (1 - np.abs(np.random.normal(0, 0.002, n)))
    open_ = np.roll(close, 1)
    open_[0] = close[0]
    volume = np.random.randint(1000, 10000, n).astype(float)

    df_raw = pd.DataFrame({
        'open': open_, 'high': high,
        'low': low, 'close': close,
        'tick_volume': volume
    }, index=dates)
    df_raw.index.name = 'time'

    os.makedirs('data', exist_ok=True)
    df_raw.to_csv(CSV_PATH)
    print(f'✅ Demo data created: {len(df_raw)} candles')

else:
    # Load your real data
    df_raw = pd.read_csv(CSV_PATH, index_col='time', parse_dates=True)
    print(f'✅ Loaded real data: {len(df_raw)} candles')

print(f'📅 Range: {df_raw.index[0]} → {df_raw.index[-1]}')
print(f'📊 Columns: {list(df_raw.columns)}')
print(f'\nFirst 3 candles:')
df_raw.head(3)

---
## 🔍 STEP 2 — EXPLORE YOUR DATA

Before building any model, a real ML engineer **always explores the data first.**
This is called **EDA — Exploratory Data Analysis.**

> **Rule:** Never train a model on data you don't understand.

In [ ]:
# ============================================================
# EXPLORATION 1 — Basic Statistics
# ============================================================

print('=' * 55)
print('📊 XAUUSD DATASET OVERVIEW')
print('=' * 55)
print(f'Total candles       : {len(df_raw):,}')
print(f'Date range          : {df_raw.index[0].date()} → {df_raw.index[-1].date()}')
print(f'Missing values      : {df_raw.isnull().sum().sum()}')
print(f'\nPrice Statistics:')
print(f'  Highest price     : ${df_raw["high"].max():,.2f}')
print(f'  Lowest price      : ${df_raw["low"].min():,.2f}')
print(f'  Average close     : ${df_raw["close"].mean():,.2f}')
print(f'  Price range       : ${df_raw["high"].max() - df_raw["low"].min():,.2f}')
print(f'\nCandle Statistics:')
avg_candle = (df_raw['high'] - df_raw['low']).mean()
print(f'  Avg candle size   : ${avg_candle:.2f} ({avg_candle*10:.0f} pips)')
print(f'  Avg body size     : ${abs(df_raw["close"]-df_raw["open"]).mean():.2f}')

# Bullish vs Bearish ratio
bullish = (df_raw['close'] > df_raw['open']).mean()
print(f'\n  Bullish candles   : {bullish:.1%}')
print(f'  Bearish candles   : {1-bullish:.1%}')

In [ ]:
# ============================================================
# EXPLORATION 2 — Price Chart
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Chart 1 — Full Price History
axes[0].plot(df_raw.index, df_raw['close'], color='gold', linewidth=0.8)
axes[0].fill_between(df_raw.index, df_raw['low'], df_raw['high'],
                     alpha=0.15, color='gold')
axes[0].set_title('XAUUSD — Full Price History (H1)', fontsize=14, pad=10)
axes[0].set_ylabel('Price (USD)')
axes[0].grid(True, alpha=0.2)

# Chart 2 — Volume
axes[1].bar(df_raw.index, df_raw['tick_volume'],
            color='steelblue', alpha=0.7, width=0.03)
axes[1].set_title('Tick Volume Over Time', fontsize=12)
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.2)

# Chart 3 — Candle Size Distribution
candle_sizes = (df_raw['high'] - df_raw['low']) * 10  # in pips
axes[2].hist(candle_sizes, bins=80, color='orange', alpha=0.8, edgecolor='none')
axes[2].axvline(candle_sizes.mean(), color='red', linestyle='--',
                linewidth=2, label=f'Mean: {candle_sizes.mean():.0f} pips')
axes[2].set_title('Candle Size Distribution (pips)', fontsize=12)
axes[2].set_xlabel('Pips')
axes[2].set_ylabel('Frequency')
axes[2].legend()
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('data/01_price_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Chart saved to data/01_price_overview.png')

In [ ]:
# ============================================================
# EXPLORATION 3 — Session Analysis (Your Trading Knowledge!)
# ============================================================
# This is WHERE your trading experience beats a regular ML student.
# You already KNOW sessions matter — now let's PROVE it with data.

df_explore = df_raw.copy()
df_explore['hour'] = df_explore.index.hour
df_explore['day'] = df_explore.index.dayofweek
df_explore['candle_pips'] = (df_explore['high'] - df_explore['low']) * 10
df_explore['is_bullish'] = (df_explore['close'] > df_explore['open']).astype(int)

# Session labels
def get_session(hour):
    if 0 <= hour < 8:   return 'Asian'
    if 8 <= hour < 13:  return 'London'
    if 13 <= hour < 21: return 'New York'
    return 'After Hours'

df_explore['session'] = df_explore['hour'].apply(get_session)

# Volatility by session
session_stats = df_explore.groupby('session').agg(
    avg_pips=('candle_pips', 'mean'),
    max_pips=('candle_pips', 'max'),
    bullish_rate=('is_bullish', 'mean'),
    candle_count=('candle_pips', 'count')
).round(2)

print('📊 SESSION ANALYSIS')
print('=' * 55)
print(session_stats.to_string())

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sessions_ordered = ['Asian', 'London', 'New York', 'After Hours']
colors = ['#4a90d9', '#27ae60', '#e74c3c', '#95a5a6']

# Avg volatility per session
data = [session_stats.loc[s, 'avg_pips'] if s in session_stats.index else 0
        for s in sessions_ordered]
axes[0].bar(sessions_ordered, data, color=colors, alpha=0.85)
axes[0].set_title('Avg Candle Size (Pips) by Session', fontsize=11)
axes[0].set_ylabel('Pips')
axes[0].grid(True, alpha=0.3, axis='y')

# Volatility by hour
hourly_vol = df_explore.groupby('hour')['candle_pips'].mean()
axes[1].bar(hourly_vol.index, hourly_vol.values, color='gold', alpha=0.85)
axes[1].axvspan(8, 13, alpha=0.15, color='green', label='London')
axes[1].axvspan(13, 21, alpha=0.15, color='red', label='New York')
axes[1].set_title('Volatility by Hour of Day', fontsize=11)
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Avg Pips')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis='y')

# Bullish rate by day of week
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
daily_bull = df_explore[df_explore['day'] < 5].groupby('day')['is_bullish'].mean()
axes[2].bar(days, daily_bull.values * 100, color='mediumpurple', alpha=0.85)
axes[2].axhline(50, color='white', linestyle='--', linewidth=1, label='50% (random)')
axes[2].set_title('Bullish Rate by Day of Week (%)', fontsize=11)
axes[2].set_ylabel('Bullish %')
axes[2].set_ylim(40, 60)
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('data/02_session_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n💡 KEY INSIGHT: This is EXACTLY what ML will use as features!')
print('   Higher volatility sessions = stronger signals for the model')

---
## ⚙️ STEP 3 — FEATURE ENGINEERING

This is the **most important step**. You're converting raw OHLCV candles into **signals** the model can learn from.

Think of it like this:
- **Raw candle** = just numbers (2385.5, 2390.1, 2381.2, 2388.4)
- **Features** = meaning (RSI is overbought, London session, ATR is high, trend is up)

> Your trading knowledge = the best feature engineering guide possible.

In [ ]:
# ============================================================
# FEATURE ENGINEERING — Build Your Feature Set
# ============================================================

df = df_raw.copy()

print('Building features...')

# ---- GROUP 1: MOMENTUM INDICATORS ----
# These tell us: is price moving fast? Is it overbought?
df['rsi_14']     = ta.momentum.rsi(df['close'], window=14)
df['rsi_7']      = ta.momentum.rsi(df['close'], window=7)
df['rsi_21']     = ta.momentum.rsi(df['close'], window=21)
df['stoch_k']    = ta.momentum.stoch(df['high'], df['low'], df['close'])
df['stoch_d']    = ta.momentum.stoch_signal(df['high'], df['low'], df['close'])
df['williams_r'] = ta.momentum.williams_r(df['high'], df['low'], df['close'])
df['roc']        = ta.momentum.roc(df['close'], window=10)  # rate of change
print('  ✅ Momentum indicators done')

# ---- GROUP 2: TREND INDICATORS ----
# These tell us: what direction is the market going?
df['ema_8']      = ta.trend.ema_indicator(df['close'], window=8)
df['ema_21']     = ta.trend.ema_indicator(df['close'], window=21)
df['ema_50']     = ta.trend.ema_indicator(df['close'], window=50)
df['ema_200']    = ta.trend.ema_indicator(df['close'], window=200)
df['ema_8_21']   = df['ema_8'] - df['ema_21']     # fast vs slow cross
df['ema_21_50']  = df['ema_21'] - df['ema_50']    # medium vs slow cross
df['ema_50_200'] = df['ema_50'] - df['ema_200']   # major trend direction
df['macd']       = ta.trend.macd(df['close'])
df['macd_signal']= ta.trend.macd_signal(df['close'])
df['macd_diff']  = ta.trend.macd_diff(df['close'])  # histogram
df['adx']        = ta.trend.adx(df['high'], df['low'], df['close'])  # trend strength
print('  ✅ Trend indicators done')

# ---- GROUP 3: VOLATILITY INDICATORS ----
# These tell us: how much is market moving? Is it ranging or trending?
df['atr_14']     = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=14)
df['atr_7']      = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=7)
df['bb_upper']   = ta.volatility.bollinger_hband(df['close'])
df['bb_lower']   = ta.volatility.bollinger_lband(df['close'])
df['bb_width']   = ta.volatility.bollinger_wband(df['close'])  # band width
df['bb_pct']     = ta.volatility.bollinger_pband(df['close'])  # % position in bands
df['atr_ratio']  = df['atr_14'] / df['close'] * 100  # ATR as % of price
print('  ✅ Volatility indicators done')

# ---- GROUP 4: VOLUME INDICATORS ----
df['vol_sma_20'] = df['tick_volume'].rolling(20).mean()
df['vol_ratio']  = df['tick_volume'] / df['vol_sma_20']  # above/below avg volume
df['obv']        = ta.volume.on_balance_volume(df['close'], df['tick_volume'])
print('  ✅ Volume indicators done')

# ---- GROUP 5: PRICE ACTION FEATURES ----
# Raw candle properties — things you look at as a trader
df['candle_size']  = (df['high'] - df['low'])                           # total range
df['body_size']    = abs(df['close'] - df['open'])                      # body size
df['upper_wick']   = df['high'] - df[['open','close']].max(axis=1)     # upper shadow
df['lower_wick']   = df[['open','close']].min(axis=1) - df['low']      # lower shadow
df['wick_ratio']   = (df['upper_wick'] - df['lower_wick']) / df['candle_size'].replace(0, 0.0001)
df['body_ratio']   = df['body_size'] / df['candle_size'].replace(0, 0.0001)  # body vs wick
df['is_bullish']   = (df['close'] > df['open']).astype(int)            # green candle?
df['is_doji']      = (df['body_ratio'] < 0.1).astype(int)             # doji pattern?
print('  ✅ Price action features done')

# ---- GROUP 6: RETURNS / MOMENTUM FEATURES ----
df['return_1h']  = df['close'].pct_change(1)   # 1 hour return
df['return_4h']  = df['close'].pct_change(4)   # 4 hour return
df['return_8h']  = df['close'].pct_change(8)   # 8 hour return
df['return_24h'] = df['close'].pct_change(24)  # daily return

# Volatility of returns
df['vol_10h']    = df['return_1h'].rolling(10).std()
df['vol_24h']    = df['return_1h'].rolling(24).std()
print('  ✅ Return features done')

# ---- GROUP 7: SESSION / TIME FEATURES ----
# YOUR TRADING KNOWLEDGE — encoded as numbers for ML
df['hour']          = df.index.hour
df['day_of_week']   = df.index.dayofweek    # 0=Mon, 4=Fri
df['is_asian']      = ((df['hour'] >= 0)  & (df['hour'] < 8)).astype(int)
df['is_london']     = ((df['hour'] >= 8)  & (df['hour'] < 13)).astype(int)
df['is_ny']         = ((df['hour'] >= 13) & (df['hour'] < 21)).astype(int)
df['is_london_ny']  = ((df['hour'] >= 13) & (df['hour'] < 17)).astype(int)  # overlap!
df['is_london_open']= (df['hour'] == 8).astype(int)   # London open hour
df['is_ny_open']    = (df['hour'] == 13).astype(int)  # NY open hour
df['is_monday']     = (df['day_of_week'] == 0).astype(int)
df['is_friday']     = (df['day_of_week'] == 4).astype(int)
df['is_midweek']    = (df['day_of_week'].isin([1,2,3])).astype(int)  # Tue-Thu
print('  ✅ Session/time features done')

# ---- GROUP 8: SUPPORT & RESISTANCE FEATURES ----
# Recent highs and lows — key levels you watch as a trader
df['high_24h']   = df['high'].rolling(24).max()    # 24h high
df['low_24h']    = df['low'].rolling(24).min()     # 24h low
df['dist_high']  = (df['high_24h'] - df['close']) / df['close'] * 100  # % from high
df['dist_low']   = (df['close'] - df['low_24h'])  / df['close'] * 100  # % from low
df['range_24h']  = df['high_24h'] - df['low_24h']                       # 24h range
df['pos_in_range'] = (df['close'] - df['low_24h']) / df['range_24h'].replace(0, 0.0001)  # 0-1
print('  ✅ Support/Resistance features done')

# Count total features
raw_cols = ['open', 'high', 'low', 'close', 'tick_volume']
feature_cols_all = [c for c in df.columns if c not in raw_cols]
print(f'\n🎉 Total features created: {len(feature_cols_all)}')
print('\nAll features:')
for i, col in enumerate(feature_cols_all, 1):
    print(f'  {i:2}. {col}')

---
## 🎯 STEP 4 — CREATE YOUR TARGET (What to Predict)

In [ ]:
# ============================================================
# TARGET — Define what the model should predict
# ============================================================

# PREDICTION HORIZON — how many candles ahead to predict?
HORIZON = 4  # predict 4 hours into the future

# TARGET: Will price be higher in HORIZON hours?
# 1 = price went UP   (BUY signal)
# 0 = price went DOWN (SELL signal)
df['future_close']  = df['close'].shift(-HORIZON)
df['future_return'] = (df['future_close'] - df['close']) / df['close'] * 100
df['target']        = (df['future_close'] > df['close']).astype(int)

# Drop rows that have NaN (from indicators needing history + future rows)
df.dropna(inplace=True)

# Check class balance
buy_pct  = df['target'].mean()
sell_pct = 1 - buy_pct

print(f'🎯 Target: Price direction in {HORIZON} hours')
print(f'   BUY  signals: {df["target"].sum():,} ({buy_pct:.1%})')
print(f'   SELL signals: {(df["target"]==0).sum():,} ({sell_pct:.1%})')
print(f'   Total usable candles: {len(df):,}')

if abs(buy_pct - 0.5) < 0.05:
    print('\n✅ Classes are balanced — good! Model can learn both directions.')
else:
    print('\n⚠️  Classes slightly imbalanced — this is normal for financial data.')

# Distribution of future returns
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['future_return'], bins=80, color='gold', alpha=0.8, edgecolor='none')
ax.axvline(0, color='white', linewidth=2, label='Zero (breakeven)')
ax.axvline(df['future_return'].mean(), color='red', linewidth=2,
           linestyle='--', label=f'Mean: {df["future_return"].mean():.4f}%')
ax.set_title(f'Distribution of {HORIZON}H Returns on XAUUSD', fontsize=12)
ax.set_xlabel('Return (%)')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 🤖 STEP 5 — TRAIN YOUR FIRST ML MODELS

Now the exciting part — letting the machine find patterns in your XAUUSD data!

In [ ]:
# ============================================================
# PREPARE DATA FOR TRAINING
# ============================================================

# Select best features (removing raw price cols and target-related cols)
exclude = ['open', 'high', 'low', 'close', 'tick_volume',
           'future_close', 'future_return', 'target',
           'hour', 'day_of_week',  # encoded versions exist
           'ema_8', 'ema_21', 'ema_50', 'ema_200',  # use differences instead
           'bb_upper', 'bb_lower',  # use width and pct instead
           'vol_sma_20',  # use ratio instead
           'high_24h', 'low_24h'   # use distances instead
          ]

FEATURES = [c for c in df.columns if c not in exclude]
print(f'📊 Features used for training: {len(FEATURES)}')
print(FEATURES)

X = df[FEATURES]
y = df['target']

# ⚠️  CRITICAL RULE: NEVER shuffle financial data!
# Use TIME-BASED split only.
# Train on PAST → Test on FUTURE
# Same as your backtesting logic!

TRAIN_RATIO = 0.75  # 75% train, 25% test
SPLIT = int(len(X) * TRAIN_RATIO)
VAL_SPLIT = int(len(X) * 0.875)  # last 12.5% for validation

X_train = X.iloc[:SPLIT]
X_val   = X.iloc[SPLIT:VAL_SPLIT]
X_test  = X.iloc[VAL_SPLIT:]

y_train = y.iloc[:SPLIT]
y_val   = y.iloc[SPLIT:VAL_SPLIT]
y_test  = y.iloc[VAL_SPLIT:]

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit on train only!
X_val_sc   = scaler.transform(X_val)        # transform val/test
X_test_sc  = scaler.transform(X_test)

print(f'\n📅 Data Split:')
print(f'   Train : {len(X_train):,} candles  ({df.index[0].date()} → {df.index[SPLIT].date()})')
print(f'   Val   : {len(X_val):,} candles  ({df.index[SPLIT].date()} → {df.index[VAL_SPLIT].date()})')
print(f'   Test  : {len(X_test):,} candles  ({df.index[VAL_SPLIT].date()} → {df.index[-1].date()})')

In [ ]:
# ============================================================
# MODEL 1 — Logistic Regression (The Baseline)
# ============================================================
# Always start with the simplest model.
# If complex models can't beat this, something is wrong.

print('🔵 Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, C=0.1)
lr.fit(X_train_sc, y_train)

lr_val_acc  = accuracy_score(y_val, lr.predict(X_val_sc))
lr_test_acc = accuracy_score(y_test, lr.predict(X_test_sc))

print(f'   Val  Accuracy: {lr_val_acc:.2%}')
print(f'   Test Accuracy: {lr_test_acc:.2%}')

In [ ]:
# ============================================================
# MODEL 2 — Decision Tree (Readable Rules)
# ============================================================
# This shows you EXACTLY what rules the model learned.
# Like reading your trading rules — but discovered by data!

print('🌳 Training Decision Tree...')
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42)
dt.fit(X_train, y_train)

dt_val_acc  = accuracy_score(y_val, dt.predict(X_val))
dt_test_acc = accuracy_score(y_test, dt.predict(X_test))

print(f'   Val  Accuracy: {dt_val_acc:.2%}')
print(f'   Test Accuracy: {dt_test_acc:.2%}')

# Print the decision rules — READ THESE!
# This is the model's trading logic in plain text
print('\n📖 DECISION TREE RULES (top 3 levels):')
print('   = The patterns the model discovered in your XAUUSD data =')
print()
tree_text = export_text(dt, feature_names=FEATURES, max_depth=3)
# Print first 40 lines only
for line in tree_text.split('\n')[:40]:
    print(line)

In [ ]:
# ============================================================
# MODEL 3 — Random Forest (Ensemble Power)
# ============================================================
# 100 decision trees voting together
# More stable, less overfitting than single tree

print('🌲 Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=30,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

rf_val_acc  = accuracy_score(y_val, rf.predict(X_val))
rf_test_acc = accuracy_score(y_test, rf.predict(X_test))

print(f'   Val  Accuracy: {rf_val_acc:.2%}')
print(f'   Test Accuracy: {rf_test_acc:.2%}')

In [ ]:
# ============================================================
# MODEL 4 — XGBoost (Best for Financial Data)
# ============================================================
# Industry standard for tabular financial data
# Used by top quant firms worldwide

print('⚡ Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=20,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_val_acc  = accuracy_score(y_val, xgb.predict(X_val))
xgb_test_acc = accuracy_score(y_test, xgb.predict(X_test))

print(f'   Val  Accuracy: {xgb_val_acc:.2%}')
print(f'   Test Accuracy: {xgb_test_acc:.2%}')

---
## 📊 STEP 6 — EVALUATE ALL MODELS

In [ ]:
# ============================================================
# MODEL COMPARISON
# ============================================================

results = {
    'Logistic Regression': {'model': lr, 'X_test': X_test_sc,  'val': lr_val_acc,  'test': lr_test_acc},
    'Decision Tree':       {'model': dt, 'X_test': X_test,     'val': dt_val_acc,  'test': dt_test_acc},
    'Random Forest':       {'model': rf, 'X_test': X_test,     'val': rf_val_acc,  'test': rf_test_acc},
    'XGBoost':             {'model': xgb,'X_test': X_test,     'val': xgb_val_acc, 'test': xgb_test_acc},
}

print('=' * 55)
print('📊 MODEL COMPARISON — XAUUSD Direction Prediction')
print('=' * 55)
print(f'{"Model":<22} {"Val Acc":>10} {"Test Acc":>10}')
print('-' * 45)
for name, r in results.items():
    marker = ' ⭐' if r['test'] == max(v['test'] for v in results.values()) else ''
    print(f'{name:<22} {r["val"]:>9.2%} {r["test"]:>9.2%}{marker}')

print('\n💡 CONTEXT: In financial ML:')
print('   50% = pure random (coin flip)')
print('   52% = potentially profitable after spreads')
print('   55% = very good')
print('   60%+ = exceptional (and rare!)')

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
val_scores  = [r['val']  * 100 for r in results.values()]
test_scores = [r['test'] * 100 for r in results.values()]

x = np.arange(len(names))
w = 0.35
bars1 = ax.bar(x - w/2, val_scores,  w, label='Validation', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + w/2, test_scores, w, label='Test',       color='gold',      alpha=0.85)

ax.axhline(50, color='white', linestyle='--', linewidth=1.5, label='Random (50%)')
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Accuracy Comparison — XAUUSD', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(45, 65)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('data/03_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CONFUSION MATRIX — How does the best model make mistakes?
# ============================================================

best_preds = xgb.predict(X_test)
best_proba = xgb.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['SELL (0)', 'BUY (1)'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('XGBoost Confusion Matrix\n(How often right/wrong?)', fontsize=11)

# Confidence Threshold Analysis
thresholds = np.arange(0.50, 0.75, 0.02)
accs, trade_counts = [], []

for t in thresholds:
    mask = (best_proba > t) | (best_proba < (1 - t))
    if mask.sum() > 10:
        acc = accuracy_score(y_test[mask], best_preds[mask])
        accs.append(acc * 100)
        trade_counts.append(mask.sum())
    else:
        accs.append(None)
        trade_counts.append(0)

ax2 = axes[1]
ax2_twin = ax2.twinx()

ax2.plot(thresholds, accs, 'gold', linewidth=2.5, marker='o', markersize=5, label='Accuracy %')
ax2_twin.bar(thresholds, trade_counts, width=0.015, alpha=0.3,
             color='steelblue', label='# Trades')
ax2.axhline(50, color='white', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Confidence Threshold')
ax2.set_ylabel('Accuracy (%)', color='gold')
ax2_twin.set_ylabel('Number of Trades', color='steelblue')
ax2.set_title('Accuracy vs Confidence Threshold\n(Higher threshold = fewer but better trades)', fontsize=11)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.savefig('data/04_confusion_and_threshold.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n📊 Threshold Analysis:')
print(f'{"Threshold":<12} {"Accuracy":>10} {"# Trades":>10}')
print('-' * 35)
for t, a, tc in zip(thresholds, accs, trade_counts):
    if a: print(f'{t:<12.2f} {a:>9.1f}% {tc:>10,}')

In [ ]:
# ============================================================
# FEATURE IMPORTANCE — What did the model learn?
# ============================================================
# This tells you WHICH of your features actually matter.
# Validates your trading knowledge — did the model agree
# that sessions, ATR, RSI matter?

importances = pd.Series(xgb.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(importances)))
bars = ax.barh(importances.index[::-1], importances.values[::-1],
               color=colors[::-1], alpha=0.9)
ax.set_title('Top 20 Most Important Features\n(What the XGBoost model learned about XAUUSD)',
             fontsize=12)
ax.set_xlabel('Feature Importance Score')
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, importances.values[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('data/05_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n💡 TOP 10 FEATURES:')
print('   (Higher = model uses this feature more to make decisions)\n')
for feat, imp in importances.head(10).items():
    bar = '█' * int(imp * 500)
    print(f'   {feat:<20} {bar} {imp:.4f}')

---
## 💾 STEP 7 — SAVE YOUR MODEL

In [ ]:
# ============================================================
# SAVE — Store the trained model for later use
# ============================================================

os.makedirs('models', exist_ok=True)

# Save XGBoost model
joblib.dump(xgb, 'models/xauusd_xgb_v1.pkl')

# Save scaler (needed for Logistic Regression predictions)
joblib.dump(scaler, 'models/scaler_v1.pkl')

# Save feature list (critical — must use SAME features in live trading)
joblib.dump(FEATURES, 'models/feature_list_v1.pkl')

print('✅ Models saved!')
print('   models/xauusd_xgb_v1.pkl    ← your trained model')
print('   models/scaler_v1.pkl         ← feature scaler')
print('   models/feature_list_v1.pkl   ← feature names list')
print()
print('To load and use later:')
print('   model    = joblib.load("models/xauusd_xgb_v1.pkl")')
print('   features = joblib.load("models/feature_list_v1.pkl")')
print('   signal   = model.predict(new_data[features])')

---
## 🚀 STEP 8 — LIVE SIGNAL FUNCTION

In [ ]:
# ============================================================
# LIVE SIGNAL — How to use the model for real predictions
# ============================================================
# This is the function you'll eventually connect to MT5

def get_xauusd_signal(df_live, model, features, confidence_threshold=0.60):
    """
    Get a trading signal from the ML model.

    Parameters:
        df_live : DataFrame with latest XAUUSD OHLCV data
        model   : trained XGBoost model
        features: list of feature names to use
        confidence_threshold: minimum confidence to trade (default 60%)

    Returns:
        dict with signal, confidence, and recommendation
    """
    # Apply all features to live data (same as training)
    # [In real use, apply full feature engineering here]

    # Get latest row features
    latest_features = df_live[features].iloc[-1:]

    # Get prediction + confidence
    prediction = model.predict(latest_features)[0]
    probabilities = model.predict_proba(latest_features)[0]
    confidence = probabilities.max()

    # Build result
    signal = 'BUY' if prediction == 1 else 'SELL'
    should_trade = confidence >= confidence_threshold

    return {
        'signal'      : signal,
        'confidence'  : confidence,
        'buy_prob'    : probabilities[1],
        'sell_prob'   : probabilities[0],
        'should_trade': should_trade,
        'reason'      : f'{signal} with {confidence:.1%} confidence' if should_trade
                        else f'Low confidence ({confidence:.1%}) — skip this candle'
    }


# ---- TEST with recent data ----
print('🔮 TESTING LIVE SIGNAL ON LAST 5 CANDLES:')
print('=' * 55)

for i in range(-5, 0):
    test_df = df.iloc[:i+1] if i < -1 else df
    if len(test_df) > 200:
        result = get_xauusd_signal(test_df, xgb, FEATURES, confidence_threshold=0.55)
        actual = '✅ Correct' if (
            (result['signal'] == 'BUY'  and df['target'].iloc[i] == 1) or
            (result['signal'] == 'SELL' and df['target'].iloc[i] == 0)
        ) else '❌ Wrong'

        time = df.index[i]
        print(f'\n📅 {time}')
        print(f'   Signal     : {result["signal"]}')
        print(f'   Confidence : {result["confidence"]:.1%}')
        print(f'   Buy Prob   : {result["buy_prob"]:.1%}  |  Sell Prob: {result["sell_prob"]:.1%}')
        print(f'   Trade?     : {"YES" if result["should_trade"] else "NO — skip"}')
        print(f'   Actual     : {actual}')

---
## 📋 SUMMARY — What You Built Today

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print('=' * 60)
print('🎉 XAUUSD ML BOT — NOTEBOOK 1 COMPLETE!')
print('=' * 60)

print(f'''
📊 DATASET
   Candles processed : {len(df):,}
   Features built    : {len(FEATURES)}
   Train candles     : {len(X_train):,}
   Test candles      : {len(X_test):,}

🤖 MODELS TRAINED
   ✅ Logistic Regression  → {lr_test_acc:.2%} accuracy
   ✅ Decision Tree        → {dt_test_acc:.2%} accuracy
   ✅ Random Forest        → {rf_test_acc:.2%} accuracy
   ✅ XGBoost (best)       → {xgb_test_acc:.2%} accuracy

💾 SAVED FILES
   models/xauusd_xgb_v1.pkl
   models/scaler_v1.pkl
   models/feature_list_v1.pkl
   data/01_price_overview.png
   data/02_session_analysis.png
   data/03_model_comparison.png
   data/04_confusion_and_threshold.png
   data/05_feature_importance.png

📚 WHAT YOU LEARNED
   ✅ Data collection + cleaning (Pandas)
   ✅ Exploratory Data Analysis (EDA)
   ✅ Feature engineering (your trading knowledge → ML features)
   ✅ Target creation (what to predict)
   ✅ Time-based train/test split
   ✅ Training 4 different ML models
   ✅ Evaluating with accuracy, confusion matrix
   ✅ Confidence threshold tuning
   ✅ Feature importance analysis
   ✅ Saving and loading models
   ✅ Building a live signal function

🚀 NEXT NOTEBOOK (02):
   → Walk-forward backtesting
   → Hyperparameter tuning
   → Simulated P&L and equity curve
   → Connect to MT5 for live signals
''')

print('=' * 60)
print('📌 You are now thinking like an ML Engineer!')
print('   This project goes on your GitHub and LinkedIn.')
print('=' * 60)